In [ ]:
import requests
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import logging
from collections import defaultdict
from conjugate_normal import conjugate_normal

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Simple configuration - just change these values
API_KEY = ""  # Add your API key here
CLASS_ID = "Tobasum-Testing-Joyner-Data"  # Change this if needed
SAMPLE_SIZE = None  # Set to a number for testing (e.g., 10) or None for all data

In [ ]:
def get_score_history(user_id, current_timestamp, df_sorted):
    """Get actual score history for this user up to this point"""
    # Get all previous submissions for this user before current timestamp
    user_previous = df_sorted[
        (df_sorted['user_id'] == user_id) & 
        (df_sorted['created_at'] < current_timestamp)
    ]
    
    # Return their content scores as the score history
    if len(user_previous) > 0:
        return user_previous['content_score'].tolist()
    else:
        return []  # First submission for this user

def call_summary_api(summary_text, user_id, timestamp, df_sorted, page_slug, class_id="Tobasum-Testing-Joyner-Data"):
    url = "https://itell-api.learlab.vanderbilt.edu/score/summary"

    try:
        # Get actual score history for this user
        score_history = get_score_history(user_id, timestamp, df_sorted)
        logger.info(f"Processing user {user_id}, page {page_slug}, score history length: {len(score_history)}")

        payload = {
            "page_slug": page_slug,
            "class_id": class_id,  # Now configurable
            "score_history": score_history,
            "summary": summary_text
        }
        headers = {
            "Content-Type": "application/json",
            "API-Key": API_KEY
        }

        response = requests.request("POST", url, json=payload, headers=headers, timeout=30)
        
        # Check if request was successful
        if response.status_code == 200:
            response_dict = response.json()
            # Add metadata to the response for tracking
            response_dict['api_metadata'] = {
                'user_id': user_id,
                'timestamp': timestamp.isoformat() if hasattr(timestamp, 'isoformat') else str(timestamp),
                'page_slug': page_slug,
                'score_history': score_history,
                'status_code': response.status_code,
                'request_payload': payload
            }
            logger.info(f"Successfully processed API call for user {user_id}")
            return response_dict
        else:
            # Provide error messages based on status code
            error_msg = f"API call failed for user {user_id}. Status code: {response.status_code}"
            if response.status_code == 404:
                error_msg += f" - Class '{class_id}' might not exist"
            elif response.status_code == 401:
                error_msg += " - Authentication failed (check API key)"
            elif response.status_code == 400:
                error_msg += " - Bad request (check payload format)"
            
            logger.error(f"{error_msg}, Response: {response.text}")
            return {
                'error': True,
                'status_code': response.status_code,
                'response_text': response.text,
                'error_suggestion': f"Status {response.status_code}: {'Class not found' if response.status_code == 404 else 'Check API key' if response.status_code == 401 else 'Check request format'}",
                'api_metadata': {
                    'user_id': user_id,
                    'timestamp': timestamp.isoformat() if hasattr(timestamp, 'isoformat') else str(timestamp),
                    'page_slug': page_slug,
                    'score_history': score_history,
                    'request_payload': payload
                }
            }
            
    except requests.exceptions.Timeout:
        logger.error(f"Timeout error for user {user_id}")
        return {
            'error': True,
            'error_type': 'timeout',
            'api_metadata': {
                'user_id': user_id,
                'timestamp': timestamp.isoformat() if hasattr(timestamp, 'isoformat') else str(timestamp),
                'page_slug': page_slug
            }
        }
    except requests.exceptions.RequestException as e:
        logger.error(f"Request error for user {user_id}: {str(e)}")
        return {
            'error': True,
            'error_type': 'request_exception',
            'error_message': str(e),
            'api_metadata': {
                'user_id': user_id,
                'timestamp': timestamp.isoformat() if hasattr(timestamp, 'isoformat') else str(timestamp),
                'page_slug': page_slug
            }
        }
    except Exception as e:
        logger.error(f"Unexpected error for user {user_id}: {str(e)}")
        return {
            'error': True,
            'error_type': 'unexpected',
            'error_message': str(e),
            'api_metadata': {
                'user_id': user_id,
                'timestamp': timestamp.isoformat() if hasattr(timestamp, 'isoformat') else str(timestamp),
                'page_slug': page_slug
            }
        }

In [ ]:
# Load the data from server
df = pd.read_csv("summaries.csv")

# Display basic info about the data
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Unique users: {df['user_id'].nunique()}")
print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")

# Show sample data
display(df.head())

# Prepare data for API calls
required_cols = ['user_id', 'text', 'created_at'] 
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
    print("Available columns: {list(df.columns)}")
else:
    print("All required columns present")
    
# Sort by timestamp
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.sort_values('created_at')
print(f"Rows after sorting: {len(df)}")

Total rows: 994
Columns: ['id', 'text', 'condition', 'user_id', 'page_slug', 'is_passed', 'containment_score', 'similarity_score', 'content_score', 'content_threshold', 'is_excellent', 'created_at', 'updated_at']
Unique users: 127
Date range: 2025-06-02T15:40:05.637001+00:00 to 2025-07-21T19:10:20.558299+00:00


,id,text,condition,user_id,page_slug,is_passed,containment_score,similarity_score,content_score,content_threshold,is_excellent,created_at,updated_at
0,60,Thi spage talks about how contril stuctues are...,random_reread,226ajlpobyjbksx534h6rrg2jq,3-1-control-structures,True,0.0000,0.736030,0.170597,0.004953,True,2025-06-02T15:40:05.637001+00:00,2025-06-02T15:40:05.637001+00:00
1,61,"In this chapter of the unit 3 lessons, the art...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-2-conditionals,True,0.0000,0.520184,0.077714,0.004953,True,2025-06-02T17:31:34.022806+00:00,2025-06-02T17:31:34.022806+00:00
2,62,"In this section of the lesson for unit 3, the ...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-3-loops,True,0.0000,0.779228,0.349814,0.004953,True,2025-06-02T17:36:51.540634+00:00,2025-06-02T17:36:51.540634+00:00
3,63,"In this chapter of unit 3, the article talks a...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-4-functions,True,0.0000,0.758652,0.611322,0.004953,True,2025-06-02T17:41:39.63132+00:00,2025-06-02T17:41:39.63132+00:00
4,64,"In this article for unit 3, the article talks ...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-5-error-handling,True,0.1053,0.785503,0.431579,0.004953,True,2025-06-02T17:50:59.201884+00:00,2025-06-02T17:50:59.201884+00:00


All required columns present
Rows after sorting: 994


In [ ]:
# Test API connection and class availability

def test_api_connection():
    """Test API connection with a simple request"""
    try:
        test_response = call_summary_api(
            summary_text="This is a test summary to check API connectivity.",
            user_id="test_user_123",
            timestamp=pd.Timestamp.now(),
            df_sorted=pd.DataFrame(),  # Empty df for test
            page_slug="test-page",
            class_id=CLASS_ID
        )
        
        if test_response.get('error'):
            print(f"API Test Failed: {test_response.get('error_suggestion', 'Unknown error')}")
            if test_response.get('status_code') == 404:
                print(f"Create class '{CLASS_ID}' or use an existing class")
            elif test_response.get('status_code') == 401:
                print("Check your API_KEY variable")
            return False
        else:
            print("API Test Successful!")
            print(f"Class: {CLASS_ID}")
            print(f"Response contains: {list(test_response.keys())}")
            return True
            
    except Exception as e:
        print(f"API Test Error: {str(e)}")
        return False

test_api_connection()

In [ ]:
# Sample data if needed
if SAMPLE_SIZE is not None:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)
    print(f"Testing with {len(df)} sampled entries")
else:
    print(f"Testing with all {len(df)} entries")

In [ ]:
print("Making API calls...")
logger.info(f"Starting API calls for {len(df)} submissions")

all_responses = []

for i, row in enumerate(df.itertuples()):
    print(f"Processing summary {i+1}/{len(df)} for user {row.user_id}")
    
    summary_text = row.text
    user_id = row.user_id
    timestamp = row.created_at
    
    # Call API with actual score history for this user
    response = call_summary_api(summary_text, user_id, timestamp, df, row.page_slug, CLASS_ID)
    
    if response:
        all_responses.append(response)
    
    # Small delay
    import time
    time.sleep(0.1)

print(f"Completed {len(all_responses)} API calls")

# Save to JSONL file
output_filename = "real_summary_scores_complete.jsonl"
with open(output_filename, "w") as f:
    for response in all_responses:
        f.write(json.dumps(response) + "\n")

print(f"Saved API responses to {output_filename}")
logger.info(f"Saved {len(all_responses)} responses to {output_filename}")

Initial volume threshold: 0.3410
Starting API simulation with 994 submissions...


In [ ]:
# Load and process API responses

print("Loading API responses...")

try:
    # Load responses
    with open("real_summary_scores_complete.jsonl", "r") as f:
        responses = [json.loads(line) for line in f]
    
    print(f"Loaded {len(responses)} responses")
    
    # Filter successful responses
    successful_responses = [r for r in responses if not r.get('error', False)]
    print(f"Found {len(successful_responses)} successful responses")
    
    # Process successful responses for analysis
    if len(successful_responses) > 0:
        # Extract metrics from the responses
        metrics_list = []
        for response in successful_responses:
            if 'metrics' in response:
                metric_data = response['metrics'].copy()
                # Add metadata for tracking
                if 'api_metadata' in response:
                    metric_data.update({
                        'user_id': response['api_metadata']['user_id'],
                        'timestamp': response['api_metadata']['timestamp'],
                        'page_slug': response['api_metadata']['page_slug'],
                        'score_history_length': len(response['api_metadata'].get('score_history', []))
                    })
                metrics_list.append(metric_data)
        
        if metrics_list:
            global df_metrics
            df_metrics = pd.DataFrame(metrics_list)
            print(f"Processed {len(df_metrics)} responses with metrics")
            display(df_metrics.sample(min(3, len(df_metrics))))
        else:
            print("No metrics found in responses")
    else:
        print("No successful responses to analyze")
            
except FileNotFoundError:
    print("Error: real_summary_scores_complete.jsonl not found. Please run the API calls first.")
except Exception as e:
    print(f"Error loading responses: {str(e)}")


In [ ]:
# Create visualization
if 'df_metrics' in globals() and len(df_metrics) > 0:
    # Convert timestamp to datetime for plotting
    if 'timestamp' in df_metrics.columns:
        df_metrics['timestamp'] = pd.to_datetime(df_metrics['timestamp'])
        df_metrics = df_metrics.sort_values('timestamp')
    
    sns.set_style("whitegrid")
    fig, ax = plt.subplots(1, figsize=(16, 6))

pass_rates = []

# Calculate and plot rolling average (10 submissions)
rolling_avg = df['content_score'].rolling(window=10, min_periods=1).mean()
sns.lineplot(ax=ax, x=df.index, y=rolling_avg, 
             label='Content Score Rolling Average (10 Submissions)', legend=False,
             color='blue', linewidth=1, alpha=0.6)

# Draw Observed Threshold (as the student experienced it)
sns.lineplot(ax=ax, data=df, x=df.index, y='content_threshold', 
             label='Observed Threshold in Production', legend=False,
             color='green', linewidth=1, alpha=0.6, linestyle='dashed')

# Draw Volume Prior
# updates_array = np.array(volume_updates)
# sns.lineplot(ax=ax, x=df.index, y=updates_array[:,2], 
#              label='Simulated Bayesian Volume Prior Threshold', legend=False,
#              color='purple', linewidth=1, alpha=0.6, linestyle='dashed')

# Draw Bayesian Threshold
sns.lineplot(ax=ax, data=df, x=df.index, y='bayesian_threshold', 
             label='Simulated Bayesian Personalized Threshold', legend=False,
             color='orange', linewidth=1, alpha=0.6, linestyle='dashed')

# Add points for rows with non-empty score_history
mask = df['score_history'].apply(lambda x: isinstance(x, list) and len(x) > 0)
filtered_df = df[mask]

if len(filtered_df) > 0:
    ax.scatter(filtered_df.index, filtered_df['content_score'], 
               color='red', s=30, alpha=0.7, zorder=5,
               label='Non-empty Score History')

# Manually set the threshold for the first window_size samples
# Since we don't have enough data to calculate a fair rolling average
window_size=10
rolling_20th = df["content_score"].rolling(window=window_size, min_periods=1, closed='left').quantile(0.2)
rolling_20th.iloc[:window_size] = 0.0

# Print Pass Rates
pass_rates.append({
    "Observed": (df["content_score"] > df["content_threshold"]).astype(int).mean(),
    "Simulated Bayesian Personalized": (df["content_score"] > df["bayesian_threshold"]).astype(int).mean(),
    # "Simulated Bayesian Volume Prior Only": (df["content_score"] > updates_array[:,2]).astype(int).mean(),
    "20th Percentile": (df["content_score"] > rolling_20th).astype(int).mean(),
})

# Format x-axis to show only month-year
plt.gca().xaxis.set_major_formatter(mpl.dates.DateFormatter('%b'))
plt.gca().xaxis.set_major_locator(mpl.dates.MonthLocator(interval=3))

# Rotate x-axis labels
plt.xticks(rotation=45)

# Show Pass Rates
display(pd.DataFrame(pass_rates).round(2))

# Add legend
labels_handles = {}
for ax in fig.axes:
    for handle, label in zip(*ax.get_legend_handles_labels()):
        labels_handles[label] = handle

fig.legend(labels_handles.values(), labels_handles.keys(), loc='upper right', bbox_transform=fig.transFigure, ncol=1)

plt.title('Content Score Thresholds', pad=20)
plt.xlabel('Creation Date')
plt.ylabel('Content Score')
plt.tight_layout()
    
plt.show();
